In [ ]:
"""IBM QPU ONLY — Build circuits in SF, run on real IBM hardware, fetch results."""
import os, sys, time, math, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')
IBM_TOKEN = os.getenv('IBM_QUANTUM_TOKEN', '')
print(f"IBM_TOKEN: {'LOADED' if IBM_TOKEN else 'MISSING — check .env'}")
import numpy as np
import superfermion as sf
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
RESULTS = {}
print("Imports OK.")


In [ ]:
print("=" * 60)
print("  BUILD CIRCUITS (SF)")
print("=" * 60)

def bell():
    c = sf.Circuit(2); c.h(0); c.cx(0, 1); return c

def ghz(n=3):
    c = sf.Circuit(n); c.h(0)
    for i in range(n - 1): c.cx(i, i + 1)
    return c

def qaoa(n=4):
    gamma = [0.3, 0.5]; beta = [0.2, 0.4]
    c = sf.Circuit(n)
    for q in range(n): c.h(q)
    for p in range(2):
        for i in range(n - 1):
            c.cx(i, i + 1); c.rz(2 * gamma[p], i + 1); c.cx(i, i + 1)
        for q in range(n): c.rx(2 * beta[p], q)
    return c

def grover(target='11'):
    c = sf.Circuit(2); c.h(0); c.h(1)
    if target == '11':
        c.cz(0, 1)
    elif target == '10':
        c.x(1); c.cz(0, 1); c.x(1)
    elif target == '01':
        c.x(0); c.cz(0, 1); c.x(0)
    elif target == '00':
        c.x(0); c.x(1); c.cz(0, 1); c.x(0); c.x(1)
    c.h(0); c.h(1); c.x(0); c.x(1); c.cz(0, 1); c.x(0); c.x(1); c.h(0); c.h(1)
    return c

def qft(n=4):
    c = sf.Circuit(n)
    for j in range(n):
        c.h(j)
        for k in range(j + 1, n): c.cp(math.pi / (2 ** (k - j)), k, j)
    for i in range(n // 2): c.swap(i, n - 1 - i)
    return c

CIRCUITS = [
    ("Bell", bell()),
    ("GHZ-3", ghz(3)),
    ("GHZ-5", ghz(5)),
    ("Grover-11", grover('11')),
    ("QAOA-4", qaoa(4)),
    ("QFT-4", qft(4)),
]
for name, c in CIRCUITS:
    print(f"  {name:14s}  {c.n_qubits}q  {c.gate_count} gates")
print(f"\n  {len(CIRCUITS)} circuits ready.")


In [ ]:
print("=" * 60)
print("  CONNECT TO IBM QUANTUM + PICK BACKEND")
print("=" * 60)

if not IBM_TOKEN:
    print("  No token — cannot connect.")
else:
    service = QiskitRuntimeService(channel="ibm_quantum_platform", token=IBM_TOKEN)
    backends = service.backends()
    print(f"  Backends found: {len(backends)}")
    info = []
    for b in backends:
        try:
            nq = b.num_qubits
            st = str(b.status().status) if b.status() else '?'
            pj = getattr(b.status(), 'pending_jobs', 0) if b.status() else 0
            print(f"    {b.name:<22s} {nq:3d}q  pending={pj}  status={st}")
            info.append({"name": b.name, "qubits": nq, "pending": pj, "status": st})
        except Exception as e:
            print(f"    {b.name:<22s}  (error: {str(e)[:30]})")
    if info:
        best = max(info, key=lambda x: x['qubits'])
        TARGET = best['name']
        print(f"\n  => TARGET: {TARGET} ({best['qubits']}q)")
    else:
        TARGET = "ibm_fez"
        print(f"\n  => TARGET (fallback): {TARGET}")
    RESULTS['target'] = TARGET


In [ ]:
print("=" * 60)
print(f"  NOISE DATA: {TARGET}")
print("=" * 60)

if not IBM_TOKEN:
    print("  No token — skipping.")
else:
    be = service.backend(TARGET)
    props = be.properties()
    nq = min(be.num_qubits, 10)
    print(f"  {'Qubit':<6s} {'T1(us)':<10s} {'T2(us)':<10s} {'RO err':<10s} {'CX err':<10s}")
    print("  " + "-" * 48)
    for i in range(nq):
        t1 = props.t1(i) * 1e6 if props.t1(i) else 0
        t2 = props.t2(i) * 1e6 if props.t2(i) else 0
        ro = props.readout_error(i) if hasattr(props, 'readout_error') else 0
        try:
            cx_err = props.gate_error('cx', [i, (i+1) % nq])
        except:
            cx_err = 0
        print(f"  {i:<6d} {t1:<10.1f} {t2:<10.1f} {ro:<10.4f} {cx_err:<10.4f}")


In [ ]:
print("=" * 60)
print(f"  SUBMIT & FETCH — {len(CIRCUITS)} circuits on {TARGET}")
print("=" * 60)

if not IBM_TOKEN:
    print("  No token — skipping.")
else:
    from superfermion.bridge import to_qiskit
    ibmq_be = service.backend(TARGET)
    pm = generate_preset_pass_manager(optimization_level=3, backend=ibmq_be)
    JOBS = {}
    SHOTS = 4096
    for cname, c_sf in CIRCUITS:
        print(f"\n  {'─' * 56}")
        print(f"  {cname}")
        print(f"  {'─' * 56}")
        try:
            qc = to_qiskit(c_sf)
            qc.measure_all()
            print(f"    Qiskit: {qc.num_qubits}q, depth={qc.depth()}")
            isa = pm.run(qc)
            print(f"    ISA:    depth={isa.depth()}, ops={isa.count_ops()}")
            sampler = Sampler(mode=ibmq_be)
            job = sampler.run([isa], shots=SHOTS)
            jid = job.job_id()
            print(f"    Job ID: {jid}")
            t0 = time.perf_counter()
            try:
                res = job.result(timeout=300)
                dt = time.perf_counter() - t0
                pub = res[0]
                if hasattr(pub.data, 'meas'):
                    counts = pub.data.meas.get_counts()
                elif hasattr(pub.data, 'c'):
                    counts = pub.data.c.get_counts()
                else:
                    counts = pub.data[next(iter(pub.data._fields))].get_counts()
                total = sum(counts.values())
                print(f"    Done in {dt:.1f}s, {total} shots")
                for bs, cnt in sorted(counts.items(), key=lambda x: -x[1]):
                    bar = chr(9608) * max(1, int(cnt / total * 40))
                    print(f"      {bs}: {cnt:5d} ({cnt/total*100:5.1f}%) {bar}")
                JOBS[cname] = {'job_id': jid, 'counts': counts, 'latency_s': dt, 'status': 'DONE'}
            except Exception as e:
                dt = time.perf_counter() - t0
                st = str(job.status())
                print(f"    Timeout after {dt:.1f}s: {e}")
                print(f"    Job running (status={st}) — check IBM dashboard")
                JOBS[cname] = {'job_id': jid, 'status': st, 'error': str(e)[:120]}
        except Exception as e:
            print(f"    FAILED: {e}")
            JOBS[cname] = {'error': str(e)[:120]}
    RESULTS['jobs'] = JOBS


In [ ]:
print("=" * 60)
print("  IBM QPU RESULTS — SUMMARY")
print("=" * 60)

jobs = RESULTS.get('jobs', {})
print(f"\n  Target: {RESULTS.get('target', '?')}")
print(f"  Circuits submitted: {len(jobs)}")
print(f"\n  {'Circuit':<14s} {'Job ID':<38s} {'Status':<12s} {'Latency':<10s}")
print("  " + "-" * 78)
for cname, j in jobs.items():
    jid = j.get('job_id', '—')[:36]
    st = j.get('status', 'ERROR')
    lt = f"{j.get('latency_s', 0):.1f}s" if 'latency_s' in j else '—'
    print(f"  {cname:<14s} {jid:<38s} {st:<12s} {lt:<10s}")

print(f"\n  TOP BITSTRINGS:")
for cname, j in jobs.items():
    counts = j.get('counts', {})
    if counts:
        total = sum(counts.values())
        top3 = sorted(counts.items(), key=lambda x: -x[1])[:3]
        parts = [f"{bs}={cnt/total*100:.1f}%" for bs, cnt in top3]
        print(f"    {cname:<14s}  {', '.join(parts)}")

out = ROOT / 'notebooks' / 'ibm_qpu_results.json'
with open(out, 'w') as f:
    json.dump(RESULTS, f, indent=2, default=str)
print(f"\n  Saved: {out}")
print("DONE.")
